# Chapter 4: Finite Fields - Examples

Jupyter notebook demonstrating construction of finite fields $\mathbb{F}_{p^n}$, trace and norm, and the multiplicative group.

## 1. Constructing F_4 = Z_2[x]/(x^2 + x + 1)

Elements: $0, 1, \alpha, \alpha+1$ where $\alpha^2 = \alpha + 1$. We represent elements as integers 0..3 where bit 0 is coefficient of 1 and bit 1 is coefficient of $\alpha$.

In [ ]:
def gf4_mul(a, b):
    """Multiplication in F_4 with defining polynomial x^2+x+1 = 0b111.
    Represent elements as 2-bit polynomials."""
    mod = 0b111  # x^2 + x + 1
    # Russian peasant: shift-add with reduction at degree 2
    result = 0
    while b:
        if b & 1:
            result ^= a
        carry = a >> 1
        a = (a << 1) & 0b11  # keep only low bits (0, alpha)
        if carry:
            a ^= mod & 0b11  # reduce x^2 = x + 1 => 0b11 XOR
        b >>= 1
    return result

def name(e):
    names = {0: "0", 1: "1", 2: "alpha", 3: "alpha+1"}
    return names.get(e, str(e))

print("Multiplication table for F_4:")
els = [0,1,2,3]
print("   " + "  ".join(f"{name(e):>8}" for e in els))
for a in els:
    print(f"{name(a):>3} " + "  ".join(f"{name(gf4_mul(a,b)):>8}" for b in els))

## 2. The multiplicative group is cyclic

Verify that $\mathbb{F}_q^\times$ is cyclic for $\mathbb{F}_5, \mathbb{F}_7, \mathbb{F}_9$.

In [ ]:
import math

def is_generator(g, p):
    # compute order of g in Z_p^\times
    seen = 1
    val = g
    while val != 1:
        val = val * g % p
        seen += 1
    return seen == p - 1

for p in [5, 7, 11, 13]:
    gens = [g for g in range(2, p) if is_generator(g, p)]
    print(f"F_{p} generators: {gens}")

## 3. F_9 = Z_3[x]/(x^2 + 1) = Z_3[i]

Elements are of the form $a + b\,i$ where $i^2 = -1 = 2$ in $\mathbb{Z}_3$.

In [ ]:
def f9_add(a, b):
    return ((a % 3, (a//3) % 3)[0] + (b % 3)) % 3, ((a % 3, (a//3))[1] + (b//3)) % 3

def f9_mul_raw(a, b):
    """Multiply (a0+a1 i)*(b0+b1 i) in Z_3[i] with i^2 = 2."""
    a0, a1 = a % 3, a // 3
    b0, b1 = b % 3, b // 3
    r0 = (a0*b0 + 2*a1*b1) % 3  # i^2 = -1 = 2
    r1 = (a0*b1 + a1*b0) % 3
    return r0, r1

def show(pair):
    a0, a1 = pair
    if a0 == 0 and a1 == 0: return "0"
    if a1 == 0: return str(a0)
    if a0 == 0: return f"{a1}i"
    return f"{a0} + {a1}i"

print("i^2 in F_9:", show(f9_mul_raw(3, 3)), "(i*i = 3*3 where 3 means 'i')")
print("Should be -1 = 2:", show(f9_mul_raw(3,3)))

## 4. AES field F_256 = Z_2[x]/(x^8+x^4+x^3+x+1)

Byte arithmetic used by the AES S-box and MixColumns.

In [ ]:
def gf_mul(a, b, mod=0x11B):
    """Multiply bytes in GF(2^8) using the AES reduction polynomial (x^8+x^4+x^3+x+1)."""
    result = 0
    low = mod & 0xFF          # 0x1B: run the reduction on the low byte
    while b:
        if b & 1:
            result ^= a
        carry = a & 0x80      # bit 7 determines whether we reduce (xtime)
        a = (a << 1) & 0xFF
        if carry:
            a ^= low
        b >>= 1
    return result

def gf_egcd(a, b, mod):
    """Extended Euclid for polynomials over GF(2^8); returns (g, x, y) with g = a*x + b*y (mod mod)."""
    if b == 0:
        return a, 1, 0
    r, q = a, 0
    while r != 0 and r.bit_length() >= b.bit_length():
        shift = r.bit_length() - b.bit_length()
        q ^= 1 << shift
        r ^= b << shift
    g, x1, y1 = gf_egcd(b, r, mod)
    return g, y1, x1 ^ gf_mul(q & 0xFF, y1)

def gf_inv(a, mod=0x11B):
    """Multiplicative inverse of a in GF(2^8) via the extended Euclidean algorithm."""
    a &= 0xFF
    if a == 0:
        raise ZeroDivisionError("0 has no inverse in GF(2^8)")
    g, x, _ = gf_egcd(a, mod, mod)
    return x & 0xFF

print(f"x * x = {gf_mul(2, 2):#04x}  (x * x = x^2 = 0x04)")
print(f"0x02 * 0x57 = {gf_mul(2, 0x57):#04x}   (AES: 0xae)")
print(f"0x03 * 0x57 = {gf_mul(3, 0x57):#04x}   (AES: 0xf9)")
print(f"0x57 * 0x13 = {gf_mul(0x57, 0x13):#04x}   (AES: 0xfe)")
print(f"Check: inv(0x57) = {gf_inv(0x57):#04x},  0x57 * inv(0x57) = {gf_mul(0x57, gf_inv(0x57)):#04x} (should be 0x01)")

## 5. Trace and Norm for F_16 / F_4

Trace: $Tr(\alpha) = \sum_{i=0}^{n/m - 1} \alpha^{q^{i}}$. Norm: $N(\alpha) = \alpha^{(q^n - 1)/(q^m - 1)}$.

In [ ]:
# Construct F_16 as GF(2^4) with polynomial x^4 + x + 1 = 0b10011
mod = 0b10011
def gf16_mul(a, b):
    result = 0
    while b:
        if b & 1:
            result ^= a
        carry = a >> 3
        a = (a << 1) & 0b1111
        if carry:
            a ^= mod
        b >>= 1
    return result

def gf16_pow(a, e):
    result = 1
    while e:
        if e & 1: result = gf16_mul(result, a)
        a = gf16_mul(a, a)
        e >>= 1
    return result

# Trace map from F_16 to F_2: Tr(alpha) = alpha + alpha^2 + alpha^4 + alpha^8
def trace(a):
    return gf16_mul(gf16_mul(gf16_mul(a,1) ^ gf16_pow(a,2), 1) ^ gf16_pow(a,4)) ^ gf16_pow(a,8) if False else a ^ gf16_pow(a,2) ^ gf16_pow(a,4) ^ gf16_pow(a,8)

def norm(a):
    # Norm over F_2: a * a^2 * a^4 * a^8 = a^15
    return gf16_pow(a, 15)

print("Trace and norm for each nonzero element of F_16 / F_2:")
for a in range(1, 16):
    print(f"a = {a:#05b}: Tr = {trace(a) & 1}, N = {norm(a):#05b}")